# AML - TIME TO EVENT - WIHTOUT COVARIATES 
This script creates a time-to-event phenotype file for electronic health record–defined acute myeloid leukemia. The earliest electronic health record corresponding to acute myeloid leukemia is considered the event. Time-to-event was calculated both from birth and from the date of the earliest electronic health record. 

In [1]:
# Load packages 
import pandas as pd 
import os 

In [2]:
# Set environment for workspace bucket 
my_bucket = os.getenv("WORKSPACE_BUCKET")

In [8]:
# Download necessary files from workspace bucket 
!gsutil cp {my_bucket}/data/covariates/observation_age_period.tsv .

Copying gs://rw-migration-aou-rw-9f879fc4/data/covariates/observation_age_period.tsv...
| [1 files][ 29.0 MiB/ 29.0 MiB]                                                
Operation completed over 1 objects/29.0 MiB.                                     


## AML TIME TO EVENT - ELECTRONIC HEALTH RECORDS

In [4]:
import pandas 
import os 

dataset_39009800_person_sql = """
SELECT DISTINCT c.concept_id 
FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
JOIN (
    SELECT CAST(cr.id as string) AS id       
    FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
    WHERE concept_id IN (140352) AND full_text LIKE '%_rank1]%'
) a 
ON (
    c.path LIKE CONCAT('%.', a.id, '.%') 
    OR c.path LIKE CONCAT('%.', a.id) 
    OR c.path LIKE CONCAT(a.id, '.%') 
    OR c.path = a.id
) 
WHERE is_standard = 1 AND is_selectable = 1

UNION DISTINCT 

SELECT DISTINCT concept_id
FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
WHERE concept_id IN (4304199) AND is_standard = 1
"""
dataset_39009800_person_df = pandas.read_gbq(
    dataset_39009800_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_39009800_person_df.head(5)

# =====================================================================================
# Access electronic health records data for acute myeloid leukaemia 
# =====================================================================================

concept_ids = tuple(dataset_39009800_person_df["concept_id"])

query = f"""                                
SELECT person_id, concept_id, entry_datetime, domain, visit_concept_id, visit_occurrence_id, age_at_event, 
FROM `{os.environ["WORKSPACE_CDR"]}.cb_search_all_events`
WHERE concept_id IN {concept_ids}
"""

aml_phenotype_occurrences = pd.read_gbq(query, dialect="standard")

Downloading:   0%|          |

## AML ELECTRONIC HEALTH RECORDS 

In [6]:
# =====================================================================================
# Access electronic health records concepts for acute myeloid leukaemia 
# =====================================================================================

concept_information = f"""
SELECT * 
FROM `{os.environ["WORKSPACE_CDR"]}.concept`
WHERE concept_id IN {concept_ids}
"""

concept_information = pandas.read_gbq(
    concept_information,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

concept_information.head(5)

concept_information.to_csv("aml_concept_information.tsv", sep="\t", index=False)
# concept_information.to_excel("aml_concept_information.xlsx", index=False)



Downloading:   0%|          |

In [ ]:
# Save electronic health record concept information 
!gsutil cp aml_concept_information.tsv {my_bucket}/data/phenotype_files/

## CREATE TIME TO EVENT  FILE 

In [ ]:
# Load packages 
import pandas as pd 
import os 

In [9]:
# =====================================================================================
# Calculate time to event and assign event status 
# =====================================================================================

aml_phenotype_occurrences['entry_datetime'] = (
    pd.to_datetime(aml_phenotype_occurrences['entry_datetime'])
)

earliest_rows = (
    aml_phenotype_occurrences
    .sort_values('entry_datetime') 
    .drop_duplicates(subset='person_id', 
                     keep='first')
)

observation_age_period = pd.read_csv("observation_age_period.tsv", sep ='\t')

earliest_rows = earliest_rows[["person_id", "concept_id", "entry_datetime"]]

earliest_rows["event"] = (
    earliest_rows["entry_datetime"].dt.date
)

time_to_event = earliest_rows.merge(observation_age_period, 
                                    on ="person_id", 
                                    how = "outer")

time_to_event["status"] = (
    time_to_event["event"]
    .apply(lambda x: 1 if pd.isna(x) else 2)
)

time_to_event["event"] = (
    time_to_event["event"]
    .fillna(time_to_event["observation_period_end_date"])
)

time_to_event["observation_period_start_date"] = (
    pd.to_datetime(time_to_event["observation_period_start_date"])
)

time_to_event["event"] = pd.to_datetime(time_to_event["event"])
time_to_event["observation_period_start_date"] = pd.to_datetime(
    time_to_event["observation_period_start_date"]
)

time_to_event["date_of_birth"] = pd.to_datetime(
    time_to_event["date_of_birth"]
)

time_to_event["time_to_event_days_from_observation"] = (
    time_to_event["event"] - time_to_event["observation_period_start_date"]
).dt.days

time_to_event["time_to_event_days_from_birth"] = (
    time_to_event["event"] - time_to_event["date_of_birth"]
).dt.days

time_to_event["age_at_observation_period_start"] = (
    (time_to_event["observation_period_start_date"] - time_to_event["date_of_birth"]
).dt.days / 365.25)

time_to_event["observation_period_start_year"] = (
    time_to_event["observation_period_start_date"].dt.year
)

time_to_event["date_of_birth_year"] = (
    time_to_event["date_of_birth"].dt.year
)

time_to_event.to_csv("aml_phenotype_time_to_event.tsv", 
                     sep ='\t', 
                     index = None)

In [13]:
time_to_event[~time_to_event['concept_id'].isna()]

,person_id,concept_id,entry_datetime,event,observation_period_start_date,observation_period_end_date,date_of_birth,age,observation_period,status,time_to_event_days_from_observation,time_to_event_days_from_birth,age_at_observation_period_start,observation_period_start_year,date_of_birth_year
0,1839811,140352,1983-12-07 06:00:00+00:00,1983-12-07,1985-01-02,2023-10-01,1969-06-15,54.294319,38.743326,2,-392,5288,15.550992,1985,1969
1,1482547,140352,1988-12-19 06:00:00+00:00,1988-12-19,1985-01-04,2023-10-01,1953-06-15,70.294319,38.737851,2,1445,12971,31.556468,1985,1953
2,2115037,140352,1996-06-17 05:00:00+00:00,1996-06-17,1995-01-27,2023-10-01,1932-06-15,91.293634,28.676249,2,507,23378,62.617385,1995,1932
3,1394415,140352,1997-04-08 00:00:00+00:00,1997-04-08,1996-08-02,2023-10-01,1967-06-15,56.295688,27.162218,2,249,10890,29.133470,1996,1967
4,3556782,138099,1997-04-11 05:00:00+00:00,1997-04-11,1995-01-13,2023-10-01,1957-06-15,66.294319,28.714579,2,819,14545,37.579740,1995,1957
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
769,4674463,140352,2023-08-27 00:00:00+00:00,2023-08-27,2017-08-08,2023-10-01,1951-06-15,72.295688,6.146475,2,2210,26371,66.149213,2017,1951
770,3460303,40483761,2023-08-29 10:01:00+00:00,2023-08-29,2009-10-12,2023-10-01,1941-06-15,82.294319,13.968515,2,5069,30025,68.325804,2009,1941
771,3348112,4079686,2023-08-29 15:30:00+00:00,2023-08-29,2022-01-12,2023-10-01,1957-06-15,66.294319,1.716632,2,594,24181,64.577687,2022,1957
772,2221504,140352,2023-09-12 04:26:28+00:00,2023-09-12,2014-10-02,2023-10-01,1971-06-15,52.295688,8.996578,2,3267,19082,43.299110,2014,1971


In [12]:
# Save the file to workspace bucket  
!gsutil cp aml_phenotype_time_to_event.tsv {my_bucket}/data/covariates/ 

Copying file://aml_phenotype_time_to_event.tsv [Content-Type=text/tab-separated-values]...
^C[0 files][  6.2 MiB/ 49.4 MiB]                                                
Caught CTRL-C (signal 2) - exiting
